In [1]:
import unpopular
import polars as pl
import lightkurve as lk
from astrocut import CutoutFactory
from astropy.coordinates import SkyCoord

In [5]:
results = pl.read_csv("results.csv")
candidates = results.filter(pl.col("result") == True)
candidates

ID,ra,dec,sector,camera,ccd,result
i64,f64,f64,i64,i64,i64,bool
234343491,38.263808,-66.966441,1,3,4,true
273827098,33.550229,-63.606436,1,3,4,true
133687709,121.185761,-37.102251,8,3,2,true
308991387,123.159444,-65.12131,8,4,1,true
152493668,120.061559,-34.301882,8,2,1,true
…,…,…,…,…,…,…
77621571,239.414812,-29.571551,38,1,3,true
186692318,233.846013,-26.383401,38,1,3,true
46037024,214.237379,-23.501352,38,1,4,true


In [2]:
def make_lightcurve(target, size=32):
    tic, ra, dec, sector, camera, ccd = target.values()
    coords = SkyCoord(ra, dec, frame="icrs", unit="deg")
    
    cube_cutter = CutoutFactory()

    cube_file = f"s3://stpubdata/tess/public/mast/tess-s{str(sector).zfill(4)}-{camera}-{ccd}-cube.fits"
    cutout = cube_cutter.cube_cut(cube_file, coordinates=coords, cutout_size=size, verbose=True, threads="auto")

    s = unpopular.Source(cutout, remove_bad=True)
    s.set_aperture(rowlims=[size//2, size//2 + 1], collims=[size//2, size//2 + 1])
    
    s.add_cpm_model(exclusion_size=5, n=64, predictor_method="similar_brightness")
    s.set_regs([0.1])
    s.holdout_fit_predict(k=100)

    apt_detrended_flux = s.get_aperture_lc(data_type="cpm_subtracted_flux")
    
    os.remove(cutout)
    return lk.TessLightCurve(time=s.time, flux=apt_detrended_flux)